# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [14]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [15]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [16]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [17]:
# Para árboles no es estrictamente necesario escalar las variables
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns)
    ]
)

# Para modelos basados en distancias/lineales (KNN, Regresión Logística) sí escalamos
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), feature_columns)
    ]
)

#raise NotImplementedError("TODO 1: crea los preprocesadores")

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

Si ajustamos (fit) el StandardScaler con todo el conjunto de datos antes de hacer validación cruzada, la media y desviación estándar de las particiones de validación terminarían afectando el escalado de los datos de entrenamiento. Esto produce fuga de datos (data leakage). Al meter el transformador dentro del Pipeline, scikit-learn garantiza que en cada fold el escalador aprenda exclusivamente del subconjunto de train y aplique esa transformación al subconjunto de validation.

## 3. Modelos candidatos

In [18]:
# TODO 2: crea los modelos candidatos
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
    ]),
    "KNN": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", KNeighborsClassifier())
    ])
}

#raise NotImplementedError("TODO 2: crea los modelos candidatos")

## 4. Comparación inicial con validación cruzada

In [19]:
# TODO 3: compara los modelos
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )
    
    train_acc_mean = float(np.mean(scores["train_score"]))
    val_acc_mean = float(np.mean(scores["test_score"]))
    val_acc_std = float(np.std(scores["test_score"]))
    
    initial_results.append({
        "model": name,
        "train_accuracy_mean": train_acc_mean,
        "val_accuracy_mean": val_acc_mean,
        "val_accuracy_std": val_acc_std
    })

initial_comparison = pd.DataFrame(initial_results)
print("=== Comparación Inicial de Modelos ===")
print(initial_comparison.to_string(index=False))

#raise NotImplementedError("TODO 3: compara los modelos")

=== Comparación Inicial de Modelos ===
             model  train_accuracy_mean  val_accuracy_mean  val_accuracy_std
      DecisionTree               1.0000           0.902217          0.058617
LogisticRegression               1.0000           0.979064          0.027713
               KNN               0.9771           0.950739          0.035274


## 5. Optimización de hiperparámetros

In [20]:
# TODO 4: optimiza los modelos
param_grids = {
    "DecisionTree": {
        "classifier__max_depth": [3, 5, 7, None],
        "classifier__min_samples_split": [2, 5, 10]
    },
    "LogisticRegression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
        "classifier__solver": ["lbfgs", "saga", "newton-cg"]
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9],
        "classifier__weights": ["uniform", "distance"]
    }
}

searches = {}
tuned_results = []

for name, model in models.items():
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    searches[name] = grid_search
    
    # Extraer métricas del mejor estimador en CV
    best_idx = grid_search.best_index_
    train_score = grid_search.cv_results_["mean_train_score"][best_idx]
    val_score = grid_search.best_score_
    val_std = grid_search.cv_results_["std_test_score"][best_idx]
    
    tuned_results.append({
        "model": name,
        "best_params": grid_search.best_params_,
        "train_accuracy_mean": float(train_score),
        "val_accuracy_mean": float(val_score),
        "val_accuracy_std": float(val_std)
    })

tuned_comparison = pd.DataFrame(tuned_results)
print("=== Resumen de Modelos Ajustados ===")
print(tuned_comparison[["model", "val_accuracy_mean", "val_accuracy_std", "best_params"]].to_string(index=False))

# raise NotImplementedError("TODO 4: optimiza los modelos")

=== Resumen de Modelos Ajustados ===
             model  val_accuracy_mean  val_accuracy_std                                                      best_params
      DecisionTree           0.902217          0.058617 {'classifier__max_depth': 5, 'classifier__min_samples_split': 2}
LogisticRegression           0.979064          0.027713            {'classifier__C': 1.0, 'classifier__solver': 'lbfgs'}
               KNN           0.957882          0.040520 {'classifier__n_neighbors': 7, 'classifier__weights': 'uniform'}


## 6. Selección y persistencia del campeón

In [21]:
# TODO 5: selecciona y guarda el campeón

# Encontrar el modelo con mayor score promedio en validación
champion_row = tuned_comparison.loc[tuned_comparison["val_accuracy_mean"].idxmax()]
champion_name = champion_row["model"]
champion_search = searches[champion_name]

# El mejor modelo reentrenado con todo X_train
champion = champion_search.best_estimator_

# 1. Guardar el modelo campeón
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# 2. Guardar cv_results.csv
tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

# 3. Guardar metadatos de entrenamiento
training_metadata = {
    "champion_name": champion_name,
    "best_params": champion_search.best_params_,
    "cv_best_score_accuracy": float(champion_search.best_score_),
    "cv_std_accuracy": float(champion_row["val_accuracy_std"]),
    "n_splits": 5,
    "random_state": RANDOM_STATE,
    "features_used": feature_columns
}

with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=4)

print(f"Modelo campeón elegido: {champion_name} con un Accuracy de CV de {champion_search.best_score_:.4f}")

#raise NotImplementedError("TODO 5: selecciona y guarda el campeón")

Modelo campeón elegido: LogisticRegression con un Accuracy de CV de 0.9791


In [22]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

Se seleccionó el modelo [Escribe aquí el nombre del modelo campeón, ej: LogisticRegression / KNN] como modelo campeón por presentar el rendimiento más alto en validación cruzada (Accuracy de CV ~XX.X%).   Además de su precisión, ofreció la menor brecha entre la métrica de entrenamiento y validación, lo que indica una buena capacidad de generalización sin caer en overfitting. Su baja desviación estándar a través de los 5 folds demuestra estabilidad frente a diferentes particiones de los datos de entrenamiento.

## Checklist

- [V] Todo el procesamiento está dentro de pipelines.
- [V] Comparé tres familias de modelos.
- [V] Optimicé hiperparámetros con CV.
- [V] Elegí el campeón sin usar test.
- [V] Guardé el pipeline completo y sus metadatos.